Copyright 2026 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License

In [ ]:
# @title Install Distributed Graph Flow (DGF)
!pip install -U uv pip
!uv pip install -q -U dgf

In [ ]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import dgf

In [ ]:
# @title Spanner Graph Details for Training

project_id =  "PROJECT_ID"
instance_id =  "INSTANCE_ID"
database_id =  "DATABASE_ID"
graph_id_train = "TRAIN_GRAPH_ID"
model_dir = "gs://MODEL_BUCKET/..."

In [ ]:
# @title Load Spanner Graph for Training

graph_train, schema = dgf.io.read_spanner_graph(
    project=project_id,
    instance=instance_id,
    database=database_id,
    graph=graph_id_train,
)

In [ ]:
# @title Feature Prep

## No features on edges
for edge_set in schema.edge_sets.values():
  edge_set.features = {}


## Node Set - event
node_set = schema.node_sets["event"]
node_set.features["event_id"].semantic = dgf.data.FeatureSemantic.PRIMARY_ID
node_set.features["incident_type"].semantic = (
    dgf.data.FeatureSemantic.CATEGORICAL
)
node_set.features["root_cause"].semantic = dgf.data.FeatureSemantic.CATEGORICAL
del node_set.features["change_request_id"]
del node_set.features["t_start"]
del node_set.features["t_end"]
del node_set.features["ttr_minutes"]


## Node Set - anomaly
node_set = schema.node_sets["anomaly"]
node_set.features["anomaly_id"].semantic = dgf.data.FeatureSemantic.PRIMARY_ID
node_set.features["anomaly_type"].semantic = (
    dgf.data.FeatureSemantic.CATEGORICAL
)
node_set.features["anomaly_label"].semantic = (
    dgf.data.FeatureSemantic.CATEGORICAL
)
node_set.features["criticality"].semantic = dgf.data.FeatureSemantic.CATEGORICAL
node_set.features["impact_score"].semantic = dgf.data.FeatureSemantic.NUMERICAL
del node_set.features["composite_anomaly_score"]


## Node Set - entity
node_set = schema.node_sets["entity"]
node_set.features["eid"].semantic = dgf.data.FeatureSemantic.PRIMARY_ID
node_set.features["network_domain"].semantic = (
    dgf.data.FeatureSemantic.CATEGORICAL
)
node_set.features["network_segment"].semantic = (
    dgf.data.FeatureSemantic.CATEGORICAL
)
node_set.features["entity_type"].semantic = dgf.data.FeatureSemantic.CATEGORICAL
node_set.features["criticality_tier"].semantic = (
    dgf.data.FeatureSemantic.CATEGORICAL
)
for feature_name, feature in node_set.features.items():
  if feature_name.startswith("time_series"):
    feature.semantic = dgf.data.FeatureSemantic.TIMESERIES
for feature_name, feature in node_set.features.items():
  if (
      feature_name.startswith("avg")
      or feature_name.startswith("min")
      or feature_name.startswith("max")
      or feature_name.startswith("median")
      or feature_name.startswith("var")
      or feature_name.startswith("stddev")
  ):
    feature.semantic = dgf.data.FeatureSemantic.NUMERICAL

In [ ]:
dgf.plot.plot_schema(schema, features=False)

In [ ]:
dgf.analyse.print_schema(schema)

In [ ]:
model = dgf.learning.train_node_model(
    graph=graph_train,
    schema=schema,
    target_nodeset="anomaly",
    target_column="impact_score",
    num_train_steps=3000
)

In [ ]:
model.describe()

In [ ]:
model.evaluate(graph_train)

In [ ]:
model.save(model_dir)

In [ ]:
# @title Spanner Graph Details for Test

project_id =  "PROJECT_ID"
instance_id =  "INSTANCE_ID"
database_id =  "DATABASE_ID"
graph_id_test = "TEST_GRAPH_ID"
model_dir = "gs://MODEL_BUCKET/..."

In [ ]:
# @title Load Spanner Graph for Test

graph_test, _ = dgf.io.read_spanner_graph(
    project=project_id,
    instance=instance_id,
    database=database_id,
    graph=graph_id_test,
)

In [ ]:
model = dgf.learning.load_model(model_dir)

In production workflows, the `model.predict` will be called on the new data inserted into the graph. Here we use one of the dates already available in the dataset.

In [ ]:
# Prediction date from test
pred_date = '20260612'

In [ ]:
## we extract the node indexes of the anomaly seed nodes
pred_pattern_str = f'ANOM-SYN-{pred_date}'
pred_pattern = np.bytes_(pred_pattern_str.encode('utf-8'))
anomaly_ndarray = graph_test.node_sets["anomaly"].features["anomaly_id"]
mask_pred = np.strings.startswith(anomaly_ndarray, pred_pattern)
indices_pred = np.flatnonzero(mask_pred)
print("indexes for the test seed nodes : ", indices_pred)

In [ ]:
# @title model.predict

if indices_pred.size > 0:
  pred_anom = model.predict(graph_test, seed_node_idxs=indices_pred)
else:
  raise ValueError(f"No anomalies found for date {pred_date}")

In [ ]:
# Return pred_anom predicted impact score along with respective entity IDs
print("Predicted Impact Scores for the seed nodes : " , pred_anom)

In [ ]:
# @title Find the corresponding Anomaly node with the highest impact score

orig_anomaly_index = indices_pred[np.argmax(pred_anom)]
orig_anomaly_id = graph_test.node_sets["anomaly"].features["anomaly_id"][orig_anomaly_index]
print(f"Anomaly Related to the Origin Entity : {orig_anomaly_id}")

In [ ]:
# @title Find the corresponding affected entity nodes

adj_anomaly_affects_entity = graph_test.edge_sets['anomaly_affects_entity'].adjacency
orig_entity_indexes = adj_anomaly_affects_entity[1,adj_anomaly_affects_entity[0,:] == orig_anomaly_index]
origin_eid = graph_test.node_sets["entity"].features["eid"][orig_entity_indexes]

print(f"Origin Entity ID : {origin_eid}")